# CIFAR-10 CNN — RecPulse

Training a Convolutional Neural Network on CIFAR-10 (32x32 color images, 10 classes).

**Architecture:** Conv(3→32) → Conv(32→64) → Conv(64→128) → FC(2048→256) → FC(256→10)

With MaxPool, Dropout, and ReLU throughout.

**Note:** There is a known VRAM leak (~3.5 MB/batch) due to missing tensor reference counting. On a 16GB GPU, expect ~5 epochs before OOM. A fix (C-level refcounting) is planned.

In [ ]:
import sys
import time
sys.path.insert(0, '..')

import recpulse_cuda as rp
from recpulse.module import Module, Linear, Conv2d, MaxPool2d, Dropout
from recpulse.optim import Adam
from recpulse.scheduler import ReduceLROnPlateau
from recpulse.data import load_cifar10, get_batch_4d

rp.manual_seed(42)

DEVICE = 'cuda'

print(f"RecPulse loaded (device={DEVICE})")

## Load Data

CIFAR-10: 50K training, 10K test. 32x32 RGB images. Classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

In [ ]:
train_images, train_labels, test_images, test_labels = load_cifar10('../data/cifar10')

CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Train: {train_images.shape} ({train_images.shape[0]} images)")
print(f"Test:  {test_images.shape} ({test_images.shape[0]} images)")
print(f"Classes: {CLASSES}")

## Define CNN

Three conv blocks (conv → relu → maxpool) followed by two fully-connected layers. Dropout for regularization.

In [ ]:
class CIFAR10CNN(Module):
    def __init__(self):
        super().__init__()
        # Block 1: 3x32x32 -> 32x16x16
        self.conv1 = Conv2d(3, 32, 3, padding=1)
        self.pool1 = MaxPool2d(2)

        # Block 2: 32x16x16 -> 64x8x8
        self.conv2 = Conv2d(32, 64, 3, padding=1)
        self.pool2 = MaxPool2d(2)

        # Block 3: 64x8x8 -> 128x4x4
        self.conv3 = Conv2d(64, 128, 3, padding=1)
        self.pool3 = MaxPool2d(2)

        # Classifier: 128*4*4 = 2048 -> 256 -> 10
        self.fc1 = Linear(128 * 4 * 4, 256)
        self.drop = Dropout(0.4)
        self.fc2 = Linear(256, 10)

    def forward(self, x):
        # Conv blocks
        h = self.keep(self.conv1(x))
        h = self.keep(h.op_relu())
        h = self.keep(self.pool1(h))

        h = self.keep(self.conv2(h))
        h = self.keep(h.op_relu())
        h = self.keep(self.pool2(h))

        h = self.keep(self.conv3(h))
        h = self.keep(h.op_relu())
        h = self.keep(self.pool3(h))

        # Flatten and classify
        h = self.keep(h.reshape([x.shape[0], 128 * 4 * 4]))
        h = self.keep(self.fc1(h))
        h = self.keep(h.op_relu())
        h = self.keep(self.drop(h))
        return self.fc2(h)

model = CIFAR10CNN()
model.to(device=DEVICE)

num_params = sum(t.size for t in model.parameters())
print(f"Model parameters: {num_params:,}")
print(f"Device: {DEVICE}")

## Setup Training

Adam optimizer with ReduceLROnPlateau — drops learning rate when validation loss stalls.

In [ ]:
optimizer = Adam(model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, patience=3, factor=0.5, min_lr=1e-6)

BATCH_SIZE = 64
NUM_EPOCHS = 5  # limited by VRAM leak on 16GB GPU — increase on larger GPUs
num_train = train_images.shape[0]
num_batches = (num_train + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Optimizer: Adam (lr=0.001)")
print(f"Scheduler: ReduceLROnPlateau (patience=3, factor=0.5)")
print(f"Batch size: {BATCH_SIZE}")
print(f"Batches per epoch: {num_batches}")
print(f"Total epochs: {NUM_EPOCHS}")

## Accuracy Helper

In [ ]:
def compute_accuracy(model, images, labels, batch_size=200):
    model.eval()
    correct = 0
    total = 0

    for b in range((len(labels) + batch_size - 1) // batch_size):
        batch_img, batch_lbl = get_batch_4d(images, labels, b, batch_size)
        if batch_img is None:
            break

        if DEVICE != 'cpu':
            batch_img = batch_img.to(device=DEVICE)

        out = model(batch_img)

        if DEVICE != 'cpu':
            out = out.to(device='cpu')

        preds = out.to_numpy().argmax(axis=1)

        for i in range(len(preds)):
            if preds[i] == batch_lbl[i]:
                correct += 1
            total += 1

        del batch_img, out

    model.train()
    return correct / total

## Train

30 epochs with gradient clipping. Test accuracy evaluated every epoch.

In [ ]:
import gc
history = {'loss': [], 'acc': [], 'lr': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    num_loss = 0
    start = time.time()

    for b in range(num_batches):
        batch_img, batch_lbl = get_batch_4d(train_images, train_labels, b, BATCH_SIZE)
        if batch_img is None:
            break

        if DEVICE != 'cpu':
            batch_img = batch_img.to(device=DEVICE)

        model.zero_grad()
        out = model(batch_img)
        loss = out.op_cross_entropy_loss(batch_lbl)
        epoch_loss += loss.to(device='cpu').sum_all()
        num_loss += 1
        loss.backward()
        rp.clip_grad_norm(model.parameters(), 2.0)
        optimizer.step()
        del batch_img, out, loss

    elapsed = time.time() - start
    avg_loss = epoch_loss / num_loss
    history['loss'].append(avg_loss)
    history['lr'].append(scheduler.get_lr())

    acc = compute_accuracy(model, test_images, test_labels)
    history['acc'].append(acc)
    scheduler.step(avg_loss)
    gc.collect()

    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS}  loss={avg_loss:.4f}  acc={acc:.2%}  lr={scheduler.get_lr():.6f}  ({elapsed:.1f}s)")

## Results

In [ ]:
best_acc = max(history['acc'])
best_epoch = history['acc'].index(best_acc) + 1

print(f"Best test accuracy: {best_acc:.2%} (epoch {best_epoch})")
print(f"Final test accuracy: {history['acc'][-1]:.2%}")
print(f"Final training loss: {history['loss'][-1]:.4f}")
print()
print("Training history:")
print(f"{'Epoch':>5}  {'Loss':>8}  {'Acc':>8}  {'LR':>10}")
print("-" * 40)
for i in range(len(history['loss'])):
    print(f"{i+1:5d}  {history['loss'][i]:8.4f}  {history['acc'][i]:7.2%}  {history['lr'][i]:10.6f}")

## Save Model

In [ ]:
rp.save(model.tracked, "../data/cifar10_cnn.rpt")
print("Model saved to data/cifar10_cnn.rpt")